# Prepare CFD Dataset for Beta-VAE Training

In [ ]:
import os
import pickle
import random
from typing import List, Tuple

import torch
import matplotlib.pyplot as plt
# from PIL import Image
# from torch.utils.data import Dataset
# from torchvision import transforms

## Config

In [2]:
CFD_DIR = "/home/jhanliufu/RutishauserLab/DATA/cfd/cfd/CFD Version 3.0/Images/CFD"
OUTPUT_DIR = "../datasets"
IMAGE_SIZE = 64          # pixels (square)
# EXPRESSIONS = ["N"]      # N=neutral; options: N, A, F, HC, HO
EXPRESSIONS = ["N", "A", "F", "HC", "HO"]
SEED = 42

# Overfit set
# number of images in train; same number held out for val
N_OVERFIT = 8           

random.seed(SEED)

## Dataset utilities

In [3]:
import sys
sys.path.insert(0, "..")

from data_utils import FaceDataset, find_images

all_paths = find_images(CFD_DIR, EXPRESSIONS)
print(f"Found {len(all_paths)} images matching expressions {EXPRESSIONS}")

Found 1207 images matching expressions ['N', 'A', 'F', 'HC', 'HO']


## Overfit set — visualize then save

In [ ]:
image_size = (IMAGE_SIZE, IMAGE_SIZE)
sampled = random.sample(all_paths, N_OVERFIT * 2)
train_paths = sampled[:N_OVERFIT]
val_paths = sampled[N_OVERFIT:]

train_ds = FaceDataset(train_paths, image_size)
val_ds = FaceDataset(val_paths, image_size)

# Stack all tensors: [N, C, H, W]
train_tensors = torch.stack([train_ds[i] for i in range(len(train_ds))])
val_tensors = torch.stack([val_ds[i] for i in range(len(val_ds))])

# fig, axes = plt.subplots(2, N_OVERFIT, figsize=(N_OVERFIT * 2, 5))
# fig.suptitle(f"Overfit set — {IMAGE_SIZE}×{IMAGE_SIZE}  |  top: train  |  bottom: val", fontsize=12)

# for i in range(N_OVERFIT):
#     axes[0, i].imshow(train_tensors[i].permute(1, 2, 0))
#     axes[0, i].axis("off")
#     axes[0, i].set_title(os.path.basename(train_paths[i]).split("-N")[0], fontsize=6)

#     axes[1, i].imshow(val_tensors[i].permute(1, 2, 0))
#     axes[1, i].axis("off")
#     axes[1, i].set_title(os.path.basename(val_paths[i]).split("-N")[0], fontsize=6)

# plt.tight_layout()
# plt.show()

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

for tensor, name in [
    (train_tensors, "overfit_cfd_train.pkl"),
    (val_tensors, "overfit_cfd_val.pkl"),
]:
    path = os.path.join(OUTPUT_DIR, name)
    with open(path, "wb") as f:
        pickle.dump(tensor, f)
    print(f"Saved {len(tensor)} images → {path}")

Saved 8 images → ../datasets/overfit_cfd_train.pkl
Saved 8 images → ../datasets/overfit_cfd_val.pkl


## Full dataset

In [ ]:
VAL_SPLIT = 0.1

random.shuffle(all_paths)
n_val = max(1, int(len(all_paths) * VAL_SPLIT))
full_val_paths = all_paths[:n_val]
full_train_paths = all_paths[n_val:]

print(f"Full train: {len(full_train_paths)} images | Full val: {len(full_val_paths)} images")
print("Applying transforms and loading into memory...")

full_train_ds = FaceDataset(full_train_paths, image_size)
full_val_ds = FaceDataset(full_val_paths, image_size)

full_train_tensors = torch.stack([full_train_ds[i] for i in range(len(full_train_ds))])
full_val_tensors = torch.stack([full_val_ds[i] for i in range(len(full_val_ds))])

for tensor, name in [
    (full_train_tensors, "full_cfd_train.pkl"),
    (full_val_tensors, "full_cfd_val.pkl"),
]:
    path = os.path.join(OUTPUT_DIR, name)
    with open(path, "wb") as f:
        pickle.dump(tensor, f)
    print(f"Saved {len(tensor)} images → {path}")

Full train: 1087 images | Full val: 120 images
Applying transforms and loading into memory...
Saved 1087 images → ../datasets/full_cfd_train.pkl
Saved 120 images → ../datasets/full_cfd_val.pkl
